In previous notebook we did eda for customers, orders and order items here we will do eda for products,sellers, order reviews and order payments.

In [1]:
import numpy as np
import pandas as pd
import os

folder_path = os.getcwd()
dataset_folder_path_raw = os.path.join(folder_path,"..\\data_raw\\")
dataset_folder_path_clean = os.path.join(folder_path,"..\\data_cleaned\\")

## Products EDA

In [3]:
products_dataset_folder = os.path.join(dataset_folder_path_clean,'products_dataset.csv')
products_df = pd.read_csv(products_dataset_folder)
products_df.columns

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

* Univariate Analysis
    * Category distribution
    * Weight distribution
    * Description length distribution
    * Photos count distribution

In [8]:
# Product Category counts
category_counts = products_df['product_category_name'].value_counts()

# Weight statistics
weight_bins = pd.cut(
    products_df['product_weight_g'],
    bins=[0,500,1000,2000,5000,10000,50000]
)

weight_distribution = (
    weight_bins.value_counts()
               .sort_index()
)

# Photo quantity distribution
photo_distribution = (
    products_df['product_photos_qty']
    .value_counts()
    .sort_index()
)

print(category_counts.head(),"\n")
print(weight_distribution,"\n")
print(photo_distribution)

product_category_name
bed_bath_table     3029
sports_leisure     2867
furniture_decor    2657
health_beauty      2444
housewares         2335
Name: count, dtype: int64 

product_weight_g
(0, 500]          13307
(500, 1000]        6363
(1000, 2000]       5047
(2000, 5000]       3536
(5000, 10000]      2231
(10000, 50000]     1853
Name: count, dtype: int64 

product_photos_qty
1.0     16489
2.0      6263
3.0      3860
4.0      2428
5.0      1484
6.0       968
7.0       343
8.0       192
9.0       105
10.0       95
11.0       46
12.0       35
13.0        9
14.0        5
15.0        8
17.0        7
18.0        2
19.0        1
20.0        1
Name: count, dtype: int64 



* Bivariate Analysis
    * Photos count vs description length
  
* Interesting Checks
    * Products with zero photos

In [9]:
# Finding correlation b/n photos count and description length
corr = products_df[['product_description_lenght', 'product_photos_qty']].corr()

print(corr)

                            product_description_lenght  product_photos_qty
product_description_lenght                    1.000000            0.108745
product_photos_qty                            0.108745            1.000000


In [10]:
# Finding products where no photos are there
products_df[products_df['product_photos_qty']==0]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm


## Sellers EDA

* Univariate
    * Sellers by state
    * Sellers by city

In [11]:
sellers_dataset_folder = os.path.join(dataset_folder_path_raw,'sellers_dataset.csv')
sellers_df = pd.read_csv(sellers_dataset_folder)
sellers_df.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [19]:
sellers_count = sellers_df['seller_id'].nunique()
sellers_record_count = sellers_df.shape[0]
print(sellers_record_count,sellers_count)

3095 3095


There are unique records of the sellers in the dataset

In [24]:
# Finding distribution of sellers by state and city
sellers_by_state = sellers_df['seller_state'].value_counts()
sellers_by_city = sellers_df['seller_city'].value_counts()
print(f"Distribution of sellers by : {sellers_by_state.head(10)}")
print(f"Distribution of sellers by : {sellers_by_city.head(10)}")

Distribution of sellers by : seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
Name: count, dtype: int64
Distribution of sellers by : seller_city
sao paulo         694
curitiba          127
rio de janeiro     96
belo horizonte     68
ribeirao preto     52
guarulhos          50
ibitinga           49
santo andre        45
campinas           41
maringa            40
Name: count, dtype: int64


* Interesting Checks
    * Seller concentration
    * Marketplace dependency on few states

In [25]:
order_items_dataset = os.path.join(dataset_folder_path_clean,"order_items_dataset.csv")
order_items_df = pd.read_csv(order_items_dataset)
order_items_df.columns

Index(['Unnamed: 0', 'order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='str')

In [26]:
# Top sellers contributing to 50% of all items sold
orders_sellers =(
        order_items_df
        .groupby('seller_id')['order_id']
        .count()
        .reset_index(name='items_sold')
        .sort_values('items_sold',ascending=False)
        )

total_selling_count = orders_sellers['items_sold'].sum()
orders_sellers['cum_sum'] = orders_sellers['items_sold'].cumsum()

orders_sellers['cum_pct'] = (orders_sellers['cum_sum'] / total_selling_count)*100

orders_sellers[orders_sellers['cum_pct']<=50]

,seller_id,items_sold,cum_sum,cum_pct
1235,6560211a19b47992c3666cc44a7e94c0,2033,2033,1.804705
881,4a3ca9315b744ce9f8e9374361493884,1987,4020,3.568575
368,1f50f920176fa81dab994f9023523100,1931,5951,5.282734
2481,cc419e0650a3c5ba77189a1882b7556a,1775,7726,6.858411
2643,da8622b14eb17ae2831f4ac5b9dab84a,1551,9277,8.235242
...,...,...,...,...
2871,edb1ef5e36e0c8cd84eb3c9b003e486d,175,55492,49.260542
1756,8f2ce03f928b567e3d56181ae20ae952,175,55667,49.415890
527,2c9e548be18521d1c43cde1c582c6de8,174,55841,49.570351
1555,7e1fb0a3ebfb01ffb3a7dae98bf3238d,174,56015,49.724811


In [27]:
# Top sellers responsible for 50% of orders
orders_sellers = (
    order_items_df
    .groupby('seller_id')['order_id']
    .nunique()
    .reset_index(name='orders_handled')
    .sort_values('orders_handled', ascending=False)
)

total_orders_handled = orders_sellers['orders_handled'].sum()

orders_sellers['cum_pct'] = (
    orders_sellers['orders_handled'].cumsum() / total_orders_handled * 100
)

for pct in (25,40,50,65,80):
    top_sellers = orders_sellers[orders_sellers['cum_pct'] <= pct].shape[0]
    print(f"{top_sellers} sellers handle {pct}% of orders")

27 sellers handle 25% of orders
74 sellers handle 40% of orders
129 sellers handle 50% of orders
265 sellers handle 65% of orders
534 sellers handle 80% of orders


### Finding top sellers & from which state they are delivering

In [28]:
# Finding top sellers by state

# Step 1: Merge seller state info
seller_orders = order_items_df.merge(
    sellers_df[['seller_id', 'seller_state', 'seller_city']],
    on='seller_id',
    how='left'
)

# Step 2: Aggregate by state (orders)
# Volume based aggregation
state_summary_volume = (
    seller_orders
    .groupby('seller_state')['order_id']
    .count()
    .reset_index(name='items_sold')
    .sort_values('items_sold', ascending=False)
)

# Orders based aggregation
state_summary_orders= (
    seller_orders
    .groupby('seller_state')['order_id']
    .nunique()
    .reset_index(name='orders_count')
    .sort_values('orders_count', ascending=False)
)

# Revenue based aggregation
state_summary_revenue = (
    seller_orders
    .assign(revenue=seller_orders['price'])
    .groupby('seller_state')['revenue']
    .sum()
    .reset_index()
    .sort_values('revenue', ascending=False)
)

# Step 3: Combining multiple metrics
state_summary = (
    seller_orders
    .groupby('seller_state')
    .agg(
        orders=('order_id', 'nunique'),
        items=('order_item_id', 'count'),
        revenue=('price', 'sum')
    )
    .reset_index()
    .sort_values('revenue', ascending=False)
)

# Step 4: top sellers by state
# top_sellers = state_summary_orders[['seller_id']]

# top_seller_states = seller_orders.merge(top_sellers, on='seller_id')

# state_top_sellers = (
#     top_seller_states
#     .groupby('seller_state')['seller_id']
#     .nunique()
#     .reset_index(name='top_seller_count')
#     .sort_values('top_seller_count', ascending=False)
# )

state_summary_orders



,seller_state,orders_count
22,SP,70188
8,MG,7930
15,PR,7673
16,RJ,4353
20,SC,3667
19,RS,1989
4,DF,824
2,BA,569
6,GO,463
13,PE,406


### Order reviews analysis

In [29]:
order_reviews_dataset = os.path.join(dataset_folder_path_raw,'order_reviews_dataset.csv')
order_reviews_df = pd.read_csv(order_reviews_dataset)
order_reviews_df.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [31]:
num_orders = order_reviews_df['order_id'].nunique()
num_reviews = order_reviews_df.shape[0]
print(num_orders,num_reviews)

# There are few reviews for same orders

98673 99224


* Univariate
    * Review score distribution
    * Percentage of each rating


In [32]:
# Review score distribution
review_score_distribution = order_reviews_df['review_score'].value_counts()

# % of each rating
perc_distribution = review_score_distribution  / order_reviews_df.shape[0]

print("Review score distribution is: ",review_score_distribution)

print("Percentage distribution of each rating is :",perc_distribution)

Review score distribution is:  review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64
Percentage distribution of each rating is : review_score
5    0.577763
4    0.192917
1    0.115133
3    0.082430
2    0.031756
Name: count, dtype: float64


* Interesting Checks
    * Rating skewness

In [33]:
# Rating skewness
# Skewness of review scores
rating_skewness = order_reviews_df['review_score'].skew()

print(f"Rating skewness: {rating_skewness:.3f}")

"""
Interpretation:
skew > 0 → distribution is right-skewed (more low scores, tail on right)
skew < 0 → distribution is left-skewed (more high scores, tail on left)
skew ≈ 0 → roughly symmetric
"""

Rating skewness: -1.364


In [34]:
# Skewness by product category
reviews_products = order_reviews_df.merge(
    order_items_df[['order_id','product_id']],
    on='order_id',
    how='left'
).merge(
    products_df[['product_id','product_category_name']],
    on='product_id',
    how='left'
)

category_skew = reviews_products.groupby('product_category_name')['review_score'].skew().reset_index()
category_skew = category_skew.sort_values('review_score', ascending=False)

print(category_skew.head(10))


                            product_category_name  review_score
62  portateis_cozinha_e_preparadores_de_alimentos     -0.255751
23                            diapers_and_hygiene     -0.364092
57                               office_furniture     -0.600412
59                                       pc_gamer     -0.640039
30                          fashion_male_clothing     -0.746146
46                                 home_comfort_2     -0.752354
34                                fixed_telephony     -0.798865
58                                 party_supplies     -0.874285
4                                           audio     -0.941936
47                                   home_confort     -0.942449


### Order Payments EDA

In [37]:
order_payments_dataset = os.path.join(dataset_folder_path_raw,'order_payments_dataset.csv')
order_payments_df = pd.read_csv(order_payments_dataset)
order_payments_df.head(1)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33


* Univariate
    * Payment type distribution
    * Installment distribution
    * Payment value distribution

In [40]:
# Payment type distribution
payment_type_distribution = order_payments_df['payment_type'].value_counts()
payment_type_distribution_perc = payment_type_distribution / order_payments_df.shape[0]

# Installment distribution
payment_installments_distribution = order_payments_df['payment_installments'].value_counts()
payment_installments_distribution_perc = payment_installments_distribution / order_payments_df.shape[0]

# Define payment value bins (in Brazilian Real)
bins = [0, 50, 100, 200, 500, 1000, 5000, float('inf')]

labels = [
    '0-50',
    '50-100',
    '100-200',
    '200-500',
    '500-1000',
    '1000-5000',
    '5000+'
]

payment_values_binned = pd.cut(
    order_payments_df['payment_value'],
    bins=bins,
    labels=labels
)

payment_distribution = (
    payment_values_binned
    .value_counts()
    .sort_index()
)

print("Payment type distribution:",payment_type_distribution)
print("\nPayment type distribution percentage:",payment_type_distribution_perc)
print("\nPayment Installment distribution:",payment_installments_distribution)
print("\nPayment Installment distribution:",payment_installments_distribution_perc)
print("\nPayment value distribution:",payment_distribution)

Payment type distribution: payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Payment type distribution percentage: payment_type
credit_card    0.739224
boleto         0.190440
voucher        0.055590
debit_card     0.014718
not_defined    0.000029
Name: count, dtype: float64

Payment Installment distribution: payment_installments
1     52546
2     12413
3     10461
4      7098
10     5328
5      5239
8      4268
6      3920
7      1626
9       644
12      133
15       74
18       27
11       23
24       18
20       17
13       16
14       15
17        8
16        5
21        3
0         2
22        1
23        1
Name: count, dtype: int64

Payment Installment distribution: payment_installments
1     0.505804
2     0.119487
3     0.100697
4     0.068325
10    0.051287
5     0.050430
8     0.041083
6     0.037734
7     0.015652
9     0.006199
12    0.001280
15    0.000712
18    0.000260
11    0.

 
* Bivariate
    * Payment type vs payment value
    * Installments vs payment value

In [41]:
# Payment type vs payment value
# Aggregate by payment type
payment_type_summary = (
    order_payments_df.groupby('payment_type')
                     .agg( total_value=('payment_value', 'sum'),
                           avg_value=('payment_value', 'mean'),
                           median_value=('payment_value', 'median'),
                           num_payments=('payment_value', 'count'))
                     .sort_values('total_value', ascending=False)
                     .reset_index()
)

print(payment_type_summary)

  payment_type  total_value   avg_value  median_value  num_payments
0  credit_card  12542084.19  163.319021        106.87         76795
1       boleto   2869361.27  145.034435         93.89         19784
2      voucher    379436.87   65.703354         39.28          5775
3   debit_card    217989.79  142.570170         89.30          1529
4  not_defined         0.00    0.000000          0.00             3


In [42]:
# Payment value distribution by payment types in bins
bins = [0, 50, 100, 200, 500, 1000, 5000, float('inf')]
labels = ['0-50','50-100','100-200','200-500','500-1000','1000-5000','5000+']

order_payments_df['payment_value_bin'] = pd.cut(
    order_payments_df['payment_value'],
    bins=bins,
    labels=labels
)

payment_bin_summary = (
    order_payments_df.groupby(['payment_type','payment_value_bin'])
                     .size()
                     .unstack(fill_value=0)
)

print(payment_bin_summary)

payment_value_bin   0-50  50-100  100-200  200-500  500-1000  1000-5000  5000+
payment_type                                                                  
boleto              4055    6409     5980     2662       500        174      4
credit_card        13380   22529    24642    12756      2544        942      2
debit_card           371     483      433      198        29         15      0
voucher             3587    1287      614      235        33         13      0


In [48]:
# Installment vs payment values
installments_value_summary = (
    order_payments_df.groupby('payment_installments')
                     .agg( total_value=('payment_value', 'sum'),
                           avg_value=('payment_value', 'mean'),
                           median_value=('payment_value', 'median'),
                           num_payments=('payment_value', 'count'))
                     .sort_values('num_payments', ascending=False)
                     .reset_index()
)

print(installments_value_summary)

    payment_installments  total_value   avg_value  median_value  num_payments
0                      1   5907233.36  112.420229        73.340         52546
1                      2   1579283.03  127.228150       109.420         12413
2                      3   1491103.80  142.539317       110.420         10461
3                      4   1163907.61  163.976840       117.175          7098
4                     10   2211577.34  415.085837       239.740          5328
5                      5    961174.30  183.465222       125.970          5239
6                      8   1313423.34  307.737427       212.695          4268
7                      6    822611.81  209.849952       138.550          3920
8                      7    305157.39  187.673672       140.075          1626
9                      9    131015.92  203.440870        99.920           644
10                    12     42783.24  321.678496       198.410           133
11                    15     32970.93  445.553108       255.850 

In [45]:
# Analysing payments with 0 installmentss
order_payments_df[order_payments_df['payment_installments']==0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value,payment_value_bin
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69,50-100
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94,100-200


* Interesting Checks
    * High-value orders using installments
    * Most common payment methods

In [46]:
# High value orders using installments
# Set threshold
high_value_threshold = 1000

# Filter orders with installments > 1
high_value_installments = order_payments_df[
    (order_payments_df['payment_value'] > high_value_threshold) &
    (order_payments_df['payment_installments'] > 1)
]

# Optional: select useful columns
high_value_installments = high_value_installments[
    ['order_id', 'payment_type', 'payment_value', 'payment_installments']
]

print(high_value_installments)

                                order_id payment_type  payment_value  \
160     886b114d034f4ac1d39d964c1b2a8182  credit_card        2027.16   
167     62d9b911d7c56cf455f660eecb8ddd3a  credit_card        1002.73   
247     4ff8e28200e5a7a50b448cfaaf1f8ed3  credit_card        2288.31   
359     ce6d150fb29ada17d2082f4847107665  credit_card        1586.47   
434     e11fec6c25945565c1ef4f14fc3c03b7  credit_card        1995.69   
...                                  ...          ...            ...   
103605  21a3f15754b759c91fff4535aaeb3486  credit_card        1224.03   
103622  b0b0d3285e59abf2f6c9d7e1bf761323  credit_card        3044.12   
103718  fc20b8e282da6f3fbcdd3a3cedecb723  credit_card        3782.19   
103733  fb2dccfadca8cd6ebddc5d10ae48d1f7  credit_card        1134.44   
103783  4198c92e06d92792e49f119f659e723e  credit_card        1294.26   

        payment_installments  
160                       10  
167                       10  
247                       10  
359        

In [47]:
summary = (
    high_value_installments.groupby('payment_type')
                           .agg(
                               num_orders=('order_id', 'count'),
                               avg_value=('payment_value', 'mean'),
                               max_value=('payment_value', 'max')
                           )
                           .sort_values('avg_value', ascending=False)
                           .reset_index()
)

print(summary)

  payment_type  num_orders    avg_value  max_value
0  credit_card         871  1555.062744    6929.31


### Based on observation all the high value transactions have happened through credit card